# Pythonic Circuit-Builder API for the QDK

This notebook demonstrates [qdk-pythonic](https://github.com/brycewestheimer/qdk-pythonic),
a standalone Python package that provides native Python classes for building
quantum circuits, generating Q#, and running resource estimation — without
writing Q# strings directly.

Each example below shows the same workflow done both ways: native `qsharp`
strings and the Pythonic API. The results are identical — `qdk-pythonic`
generates valid Q# under the hood and submits it through the standard
`qsharp` pipeline.

In [ ]:
%pip install git+https://github.com/brycewestheimer/qdk-pythonic.git

## Bell State: Native Q# vs. Pythonic API

In [ ]:
# --- Native qsharp ---
import qsharp

qsharp.eval("""
    operation BellState() : Result[] {
        use q = Qubit[2];
        H(q[0]);
        CNOT(q[0], q[1]);
        let r0 = MResetZ(q[0]);
        let r1 = MResetZ(q[1]);
        [r0, r1]
    }
""")

native_results = qsharp.run("BellState()", shots=100)
print("Native qsharp:", native_results[:5])

In [ ]:
# --- qdk-pythonic ---
from qdk_pythonic import Circuit

circ = Circuit()
q = circ.allocate(2)
circ.h(q[0]).cx(q[0], q[1]).measure_all()

pythonic_results = circ.run(shots=100)
print("qdk-pythonic: ", pythonic_results[:5])

In [ ]:
# The Pythonic API provides inspection methods and code generation
print(circ.draw())
print(f"Depth: {circ.depth()}, Gates: {circ.gate_count()}")
print()
print("Generated Q#:")
print(circ.to_qsharp())

## Resource Estimation: Parameterized Sweep

Resource estimation is the QDK's primary competitive advantage. The Pythonic
API makes parameterized estimation sweeps natural — construct circuits
programmatically with Python loops instead of string interpolation.

In [ ]:
# --- Native qsharp: string interpolation ---
for n in [4, 8, 16, 32]:
    body = "\n".join(
        [f"        use q = Qubit[{n}];", f"        H(q[0]);"]
        + [f"        CNOT(q[{i}], q[{i+1}]);" for i in range(n - 1)]
        + [f"        T(q[0]);"]
    )
    qsharp.eval(f"operation GHZ_{n}() : Unit {{\n{body}\n}}")
    result = qsharp.estimate(f"GHZ_{n}()")
    print(f"Native n={n}: {result}")

In [ ]:
# --- qdk-pythonic: native Python parameterization ---
from qdk_pythonic import ghz_state

for n in [4, 8, 16, 32]:
    circ = ghz_state(n)
    q = circ.qubits
    circ.t(q[0])  # non-Clifford gate for estimation
    result = circ.estimate()
    print(f"Pythonic n={n}: {result}")

## Domain Abstractions: Condensed Matter

Beyond circuit building, `qdk-pythonic` provides domain modules that map
physics problems directly to quantum circuits. Domain scientists work in
their own language — lattices, Hamiltonians, coupling constants — and the
package handles the circuit construction and Q# generation.

In [ ]:
from qdk_pythonic.domains.condensed_matter import Chain, IsingModel, simulate_dynamics

# Three lines of domain-specific Python, zero Q# knowledge required
model = IsingModel(Chain(8), J=1.0, h=0.5)
circuit = simulate_dynamics(model, time=1.0, steps=10)

print(f"Qubits: {circuit.qubit_count()}")
print(f"Gates:  {circuit.total_gate_count()}")
print(f"Depth:  {circuit.depth()}")
print()
print(f"Gate breakdown: {circuit.gate_count()}")

In [ ]:
# Scaling analysis: how do resources grow with system size?
print(f"{'Sites':>6}  {'Gates':>8}  {'Depth':>6}")
print("-" * 24)

for n in [4, 8, 12, 16]:
    circ = simulate_dynamics(IsingModel(Chain(n), J=1.0, h=0.5), time=1.0, steps=10)
    print(f"{n:>6}  {circ.total_gate_count():>8}  {circ.depth():>6}")

## Domain Abstractions: Combinatorial Optimization

QAOA circuits for combinatorial optimization problems are constructed
from the problem graph, with no manual circuit wiring.

In [ ]:
from qdk_pythonic.domains.optimization import MaxCut, QAOA

# Petersen graph
edges = [
    (0,1),(1,2),(2,3),(3,4),(4,0),
    (5,7),(7,9),(9,6),(6,8),(8,5),
    (0,5),(1,6),(2,7),(3,8),(4,9),
]
problem = MaxCut(edges=edges, n_nodes=10)
qaoa = QAOA(problem.to_hamiltonian(), p=2)
circuit = qaoa.to_circuit(gamma=[0.5, 0.3], beta=[0.7, 0.2])

print(f"Qubits: {circuit.qubit_count()}, Gates: {circuit.total_gate_count()}, Depth: {circuit.depth()}")

## Generated Q#

Every domain circuit is a standard `Circuit` object. The generated Q# is
always available for inspection — the abstraction is transparent, not opaque.

In [ ]:
# Small example: 4-site Ising, 2 Trotter steps
small = simulate_dynamics(IsingModel(Chain(4), J=1.0, h=0.5), time=0.2, steps=2)
print(small.to_qsharp())

---

## Summary

This notebook demonstrated:

- **Side-by-side comparison** — Bell state creation and simulation with
  native `qsharp` vs. the Pythonic API, producing identical results.
- **Resource estimation sweeps** — Parameterized GHZ circuits across
  sizes [4, 8, 16, 32], showing how Python loops replace Q# string
  interpolation.
- **Domain abstractions** — Condensed matter (Ising model) and
  combinatorial optimization (MaxCut/QAOA) circuits constructed from
  domain objects, not manual gate wiring.
- **Transparent code generation** — Every circuit's Q# source is
  always available via `to_qsharp()`.

For more detail, see
[qdk_pythonic_domains.ipynb](qdk_pythonic_domains.ipynb) (all four
domain modules) and
[qdk_pythonic_interop.ipynb](qdk_pythonic_interop.ipynb) (cross-format
roundtripping and circuit composition).